<a href="https://colab.research.google.com/github/emijlop/master_thesis/blob/main/notebooks/02_identity_probes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Identity-dimension probes: which dimension drives each cluster?

**Idea.** Train one linear probe per identity dimension on **frozen LaBSE** embeddings of a **community-built** benchmark (HESEIA). Each probe learns a direction in embedding space for one dimension (age, gender, disability, race/ethnicity, …). Projecting the items of another benchmark (BBQ, from notebook 01) onto these directions gives every cluster a profile across dimensions. The highest-scoring dimension is the one that, from the community's point of view, *drives* that cluster.

**How it connects to the research questions**

| Output | Question it informs |
|---|---|
| Probe quality per dimension (§3) | Is each community-defined dimension linearly represented in LaBSE at all? A precondition for everything else |
| Similarity between probe directions (§4) | Structural validity: do e.g. gender and sexual orientation share a direction or stay separate? |
| Transfer to BBQ's own labels (§5) | RQ2: does BBQ's "gender" match HESEIA's "gender"? Low transfer means divergent operationalizations, or a language/format shift (see controls) |
| Cluster driver vs. BBQ's own category (§6) | RQ1/RQ2: a BBQ cluster labelled SES but driven by HESEIA's "profession" or "race" dimension |
| Co-activation on intersectional items (§7) | Additive vs. co-constitutive intersectionality |

**Framing:** HESEIA labels are a *reference frame*, not ground truth. They are one systematization of the background concepts (contestedness), chosen deliberately because they come from the community. Every result reads as "*from HESEIA's standpoint*, this BBQ cluster is driven by …".

### Design decisions (also in `docs/decision_log.md`)

**1. Frozen LaBSE + linear probe (main analysis), not fine-tuning.**
- Fine-tuning LaBSE on identity labels would reshape the embedding space around HESEIA's categories. Clustering in that space would be **circular**: clusters would align with categories because the model was trained to make them align. The proposal's frozen-encoder argument (§4.2.1, step 3) depends on this not happening.
- Linear probes are the standard way to ask "is concept X represented in these embeddings?". They are interpretable: a weight vector is a direction you can project onto. Following Hewitt & Liang (2019), each probe is reported with a **control task** (same probe, shuffled labels), so high accuracy can't come from the probe memorizing.
- Fine-tuning appears only as an optional comparison in §8. Its embeddings are **never** used for the clustering of notebook 01.

**2. Train on HESEIA alone, not HESEIA + WinoQueer mixed.**
- If positives for one dimension came from one dataset (WinoQueer → sexual orientation) and negatives from another, the probe would learn *which dataset* an item comes from (language, format, template), not the dimension.
- WinoQueer is also highly templated (11 template sentences, 9 queer identity groups, 60 names, 182 predicates) and covers only LGBTQ+ identities. It is better used as a **held-out check** for a sexual-orientation / gender-identity probe: does a HESEIA-trained probe recognize WinoQueer items?

**3. Explicit criteria for when results can be interpreted**, fixed before looking at cluster results:
- a dimension is used only if its cross-validated **selectivity ≥ 0.2** and **AUC ≥ 0.75** on HESEIA (§3);
- for interpreting BBQ clusters, additionally **transfer AUC ≥ 0.70** on BBQ's own harmonized labels (§5).
These thresholds are a choice; record any change in the decision log.

In [ ]:
# Setup (Colab or local), as in notebook 01
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules and not Path("master_thesis").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/emijlop/master_thesis.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "master_thesis[pipeline]"], check=True)
for p in [Path("master_thesis/src"), Path("../src")]:
    if p.exists(): sys.path.insert(0, str(p.resolve()))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from benchbias import SEED, ENCODER, heseia, identity_probes as ip
from benchbias.embed import embed

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
HESEIA_CSV = Path("data/raw/heseia/heseia.csv")   # MODIFY: path to the downloaded CSV

MIN_AUC, MIN_SELECTIVITY, MIN_TRANSFER_AUC = 0.75, 0.2, 0.70

## 1. Load HESEIA and harmonize its labels

Download the CSV from the Mozilla Data Collective ("HESEIA Sentence Bias Dataset") into `data/raw/heseia/`. Run `inspect_heseia` once, then set the column names and label map in `src/benchbias/heseia.py` (or pass them here). Native labels without a mapping keep their own name. HESEIA-only dimensions such as geographic origin or profession are exactly what RQ3 is about, so they become their own probes rather than being forced into a BBQ category.

In [ ]:
heseia.inspect_heseia(HESEIA_CSV)

In [ ]:
COLUMNS = {"text": None, "labels": None, "group": None}   # MODIFY after inspection
LABEL_MAP = {}                                            # MODIFY: native label -> harmonized construct

H = heseia.load_heseia(HESEIA_CSV, columns=COLUMNS, label_map=LABEL_MAP)
print(H.shape, "| intersectional share:", H["intersectional"].mean().round(3))
Y = ip.multi_hot(H["dimensions"])
Y.sum().sort_values(ascending=False)

In [ ]:
# Co-occurrence of dimensions in HESEIA: which intersections did the community write about?
co = Y.T @ Y
co.where(~np.eye(len(co), dtype=bool)).fillna(0).astype(int)

## 2. Embed with frozen LaBSE

In [ ]:
X = embed(H["text"], ENCODER, cache=OUT / "emb_heseia_LaBSE.npy")
groups = H["group"].to_numpy() if H["group"].notna().all() else None
X.shape, ("grouped CV" if groups is not None else "no group column: plain K-fold (check near-duplicates!)")

## 3. Probe quality: is each dimension represented, beyond what the probe could memorize?

In [ ]:
quality = ip.evaluate_probes(X, Y, groups=groups, seed=SEED)
quality["usable"] = (quality["AUC"] >= MIN_AUC) & (quality["selectivity"] >= MIN_SELECTIVITY)
quality.to_csv(OUT / "probe_quality_heseia.csv")
quality.sort_values("AUC", ascending=False)

In [ ]:
probes = ip.fit_probes(X, Y)
usable_heseia = quality.index[quality["usable"].fillna(False)].tolist()
print("usable dimensions:", usable_heseia)

## 4. Geometry: how do the dimensions relate in embedding space? (structural validity)

High cosine similarity between two probe directions means LaBSE, trained on HESEIA's items, represents the two dimensions in nearly the same way. Gender identity vs. sexual orientation is the pair to watch; your brainstorming notes ask whether they are "related or siloed".

In [ ]:
sim = ip.direction_similarity({d: probes[d] for d in usable_heseia})
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sim)), sim.index, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(sim)), sim.index, fontsize=8)
for s in ax.spines.values(): s.set_visible(False)
fig.colorbar(im, ax=ax, label="cosine similarity of probe directions", shrink=0.8)
ax.set_title("HESEIA identity directions in LaBSE", loc="left", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / "probe_direction_similarity.png", dpi=200); plt.show()
sim

## 5. Transfer check on BBQ: can HESEIA's probes be read on BBQ at all?

Each probe is applied to BBQ items (from notebook 01) and scored against BBQ's **own** harmonized labels. Intersection labels count for both parts. Two readings are possible, and the controls tell them apart:
- **low transfer because of language/format**: the transfer also fails on **EsBBQ** (Spanish, same format as BBQ) or improves after per-language centering → a technical shift, not a finding;
- **low transfer despite controls**: HESEIA's and BBQ's operationalizations of that dimension diverge → evidence for RQ2.

Both views are compared. If transfer works only on **unmasked** BBQ, the probes rely on identity words rather than stereotype content.

In [ ]:
from benchbias.schema import CROSSWALK
BBQ = {}
for v in ["masked", "unmasked"]:
    df = pd.read_parquet(OUT / f"bbq_{v}_clusters.parquet")       # written by notebook 01, §10
    Xb = np.load(OUT / f"bbq_{v}_labse.npy")
    S = ip.score(Xb, probes)
    BBQ[v] = dict(df=df, X=Xb, S=S, harmonized=df["category"].map(CROSSWALK["BBQ"]))

transfer = pd.concat({v: ip.transfer_check(BBQ[v]["S"], BBQ[v]["harmonized"]) for v in BBQ}, axis=1)
transfer

In [ ]:
# Robustness: remove the language offset (HESEIA = es, BBQ = en) and re-check transfer
Xall = np.vstack([X, BBQ["masked"]["X"]])
Xc = ip.center_by_group(Xall, ["es"] * len(X) + ["en"] * len(BBQ["masked"]["X"]))
probes_c = ip.fit_probes(Xc[: len(X)], Y)
S_c = ip.score(Xc[len(X):], probes_c)
pd.concat({"raw": transfer[("masked", "transfer_AUC")],
           "language-centered": ip.transfer_check(S_c, BBQ["masked"]["harmonized"])["transfer_AUC"]}, axis=1)

In [ ]:
usable_bbq = [d for d in usable_heseia
              if transfer.get(("masked", "transfer_AUC"), pd.Series(dtype=float)).get(d, np.nan) >= MIN_TRANSFER_AUC]
print("dimensions interpretable on BBQ (masked):", usable_bbq)
# Dimensions with no BBQ counterpart (e.g. profession, geographic_origin) have no transfer AUC.
# They can still be projected, but read them descriptively: 'how much of HESEIA's X is in this cluster'.

## 6. Cluster profiles: which dimension drives each BBQ cluster?

For each cluster from notebook 01: the mean probe score per dimension, the **driver** (highest dimension), its **margin** over the runner-up, and BBQ's own dominant category. Disagreements are the interesting rows. A small margin means the cluster is driven by no single dimension, which may itself be a sign of intersectional or mixed content.

In [ ]:
profiles = {}
for v in BBQ:
    profiles[v] = ip.cluster_profiles(BBQ[v]["S"], BBQ[v]["df"]["cluster"],
                                      native=BBQ[v]["harmonized"], usable=usable_bbq or None)
    profiles[v].to_csv(OUT / f"cluster_profiles_bbq_{v}.csv")
p = profiles["masked"]
print("driver == BBQ's own category:", (p["driver"] == p["native_dominant"]).mean().round(3))
p.sort_values("driver_margin").head(20)

In [ ]:
pd.crosstab(p["native_dominant"], p["driver"], values=p["size"], aggfunc="sum").fillna(0).astype(int)

In [ ]:
# All projected dimensions (incl. HESEIA-only ones), descriptive heatmap of cluster profiles
dims_all = [d for d in probes]
prof_all = ip.cluster_profiles(BBQ["masked"]["S"], BBQ["masked"]["df"]["cluster"], native=BBQ["masked"]["harmonized"])
M = prof_all[dims_all].sort_values(by=dims_all[0]).values
fig, ax = plt.subplots(figsize=(1 + 0.5 * len(dims_all), 6))
im = ax.imshow(M, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(dims_all)), dims_all, rotation=45, ha="right", fontsize=8); ax.set_yticks([])
ax.set_ylabel("BBQ clusters (masked)")
for s in ax.spines.values(): s.set_visible(False)
fig.colorbar(im, ax=ax, label="mean probe score", shrink=0.8)
ax.set_title("BBQ clusters seen through HESEIA's dimensions", loc="left", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / "cluster_profiles_bbq_masked.png", dpi=200); plt.show()

## 7. Intersectional items: do two dimensions fire together?

For BBQ's Race×Gender and Race×SES items, how often do both corresponding probes fire? Under a co-constitutive operationalization, both should be active. If only race fires, the second axis is present in the text but not represented as a dimension.

In [ ]:
df, S = BBQ["masked"]["df"], BBQ["masked"]["S"]
for cat in ["Race_x_gender", "Race_x_SES"]:
    print(f"\n{cat}")
    display(ip.co_activation(S, df["category"] == cat).head(5))
print("\nbaseline: single-axis Race_ethnicity items")
display(ip.co_activation(S, df["category"] == "Race_ethnicity").head(5))

## 8. Optional comparison: fine-tuned encoder

Pranav's alternative: fine-tune LaBSE on HESEIA's dimensions, then freeze it and probe. This is a legitimate **comparison**, with two guardrails:
1. Fine-tune on a HESEIA **training split** only, and evaluate probes on held-out HESEIA and on BBQ, so the probes don't simply re-read the training labels.
2. Never use the fine-tuned embeddings for the clustering of notebook 01 (circularity, see top).

What it answers: *does making the encoder identity-aware change which dimension drives BBQ's clusters?* If the drivers stay the same, the frozen results are robust. If they change, report it as sensitivity to the representation.

Lightest-weight option: [SetFit](https://github.com/huggingface/setfit), which contrastively fine-tunes a sentence-transformer on few labelled examples and works with LaBSE. It runs on a Colab GPU. Sketch:

```python
# from setfit import SetFitModel, Trainer, TrainingArguments
# model = SetFitModel.from_pretrained(ENCODER, multi_target_strategy="one-vs-rest")
# trainer = Trainer(model=model, train_dataset=train_ds, args=TrainingArguments(num_epochs=1, batch_size=32))
# trainer.train()
# X_ft = model.model_body.encode(texts, normalize_embeddings=True)   # then re-run §3, §5, §6 with X_ft
```

## 9. Interpretation scaffold

| Observation | Evidence | Validity facet |
|---|---|---|
| Dimension not linearly represented (low selectivity) | §3 | limit of the instrument, not a benchmark finding |
| Two dimensions share a direction | §4 | structural |
| HESEIA probe does not transfer to BBQ's same-named category, despite controls | §5 | substantive / contestedness (RQ2) |
| BBQ cluster driven by a different dimension than its label | §6 | substantive (RQ1/RQ2) |
| HESEIA-only dimensions (e.g. profession, geographic origin) score high in BBQ clusters | §6 heatmap | content BBQ carries but does not label (RQ3) |
| Intersectional items activate only one axis | §7 | structural (additive vs. co-constitutive) |